In [1]:
import glob
import os
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from datetime import datetime
from tqdm import tqdm
from multiprocessing import Pool

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

# from torchviz import make_dot
# from torch.utils.tensorboard import SummaryWriter

# Set the default figure size for all plots
plt.rcParams["figure.figsize"] = (9, 8) # You can change the values (width, height) as needed

## Loading dataseet

In [5]:
CHIRPS_GEOTIFF_DIR = '../../CHIRPS_daily_Uttarakhand'
ERA5_GEOTIFF_DIR = '../../ERA5_daily_Uttarakhand'
def load_geotiff(file_path):
    try:
        filename = os.path.basename(file_path)
        date_str = filename.split("_")[-1].replace(".tif", "")
        date_key = datetime.strptime(date_str, "%Y-%m-%d").date()
        data = rasterio.open(file_path)
        return date_key, data, file_path
    except Exception as e:
        print(f"Failed to load {file_path}: {e}")
        return None, None, file_path
    
# Get list of GeoTIFF files
chirps_geotiff_files = glob.glob(os.path.join(CHIRPS_GEOTIFF_DIR, "CHIRPS_daily_*.tif"))
era5_geotiff_files = glob.glob(os.path.join(ERA5_GEOTIFF_DIR, "ERA5_daily_*.tif"))

chirps_results = []
for file_path in tqdm(chirps_geotiff_files, desc="Loading CHIRPS Data"):
    chirps_results.append(load_geotiff(file_path))
era5_results = []
for file_path in tqdm(era5_geotiff_files, desc="Loading ERA5 Data"):
    era5_results.append(load_geotiff(file_path))
# Create dictionary from results
chirps_data_dict = {}
for date_key, data, file_path in chirps_results:
    if date_key is not None and data is not None:
        chirps_data_dict[date_key] = data
        # print(f"Loaded {file_path} for date {date_key}")
    else:
        print(f"Skipped {file_path} due to loading error")

era5_data_dict = {}
for date_key, data, file_path in era5_results:
    if date_key is not None and data is not None:
        era5_data_dict[date_key] = data
        # print(f"Loaded {file_path} for date {date_key}")
    else:
        print(f"Skipped {file_path} due to loading error")

Loading ERA5 Data: 100%|██████████| 14435/14435 [00:04<00:00, 3142.30it/s]


## Preparing dataset

In [5]:
chirps_data_dict = dict(sorted(chirps_data_dict.items()))
era5_data_dict = dict(sorted(era5_data_dict.items()))

chirps_dates = list(chirps_data_dict.keys())
chirps_dataset = list(chirps_data_dict.values())

era5_dates = list(era5_data_dict.keys())
era5_dataset = list(era5_data_dict.values())

In [6]:
def reproject_data(src_dataset, ref_shape, ref_dtype, ref_transform, ref_crs, ref_nodata):
    reprojected = []
    n = 0
    for src in tqdm(src_dataset, desc = "Reprojecting images", colour="GREEN"):
        # 2. Create a destination array with the dimensions and data type of the reference
        # This is where the reprojected data will be stored
        destination_array = np.empty(ref_shape, dtype=ref_dtype)

        # 3. Perform the reprojection
        # This function does all the hard work of warping the source data to the
        # destination grid.
        reproject(
            source=rasterio.band(src, 5),
            destination=destination_array,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref_transform, # Apply transform of the reference!
            dst_crs=ref_crs,             # Apply CRS of the reference!
            dst_nodata=ref_nodata,
            resampling=Resampling.bilinear # Choose your resampling method
        )
        reprojected.append(destination_array)
        src.close()
    return reprojected

In [7]:
def fill_nan(image):
    image[np.isnan(image)] = 0.0
    return image

### Storing CHIRPS and raw ERA5 data in list

In [8]:
chirps_images = []
for data in tqdm(chirps_dataset, desc = "Storing CHIRPS images", colour="GREEN"):
    image = data.read(1) # Store precipitation band
    image = fill_nan(image)
    chirps_images.append(image)
chirps_images = np.array(chirps_images)

raw_era5_images = []
for data in tqdm(era5_dataset, desc = "Storing raw ERA5 images", colour="GREEN"):
    raw_era5_images.append(data.read(5)) # Store precipitation band
raw_era5_images = np.array(raw_era5_images)

Storing raw ERA5 images:  93%|█████████▎| 13454/14435 [00:59<00:05, 174.83it/s]

: 

### Reprojecting ERA5 to CHIRPS transform and crs

In [ ]:
ref = chirps_dataset[0]
reprojected_era5 = reproject_data(era5_dataset, ref.shape, ref.dtypes[0], ref.transform, ref.crs, ref.nodata)

In [ ]:
i = 10_591
chirps_image = np.copy(chirps_images[i]) # Get precipitation band
era5_image = np.copy(raw_era5_images[i]*1000) # ERA5 data has precipitation unit in m/d, CHIRPS has in mm/d\
reprojected_era5_image = np.copy(reprojected_era5[i]*1000)

fig, ax = plt.subplots(1, 3, figsize=(12,7), dpi=200)

image_0 = ax[0].imshow(chirps_image, cmap="turbo")
ax[0].set_title(f"CHIRPS (5KM) {chirps_dates[i]}")
ax[0].set_axis_off()
divider = make_axes_locatable(ax[0])
cax = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(image_0, cax=cax, orientation="vertical", label="Precipitation (mm/d)")

image_1 = ax[1].imshow(era5_image, cmap="turbo")
ax[1].set_title(f"ERA5 (25KM) {era5_dates[i]}")
ax[1].set_axis_off()
divider = make_axes_locatable(ax[1])
cax = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(image_1, cax=cax, orientation="vertical", label="Precipitation (mm/d)")

image_2 = ax[2].imshow(reprojected_era5_image, cmap="turbo")
ax[2].set_title(f"ERA5 (25KM) Reprojected {era5_dates[i]}")
ax[2].set_axis_off()
divider = make_axes_locatable(ax[2])
cax = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(image_2, cax=cax, orientation="vertical", label="Precipitation (mm/d)");

# plt.suptitle(f"{chirps_dates[i]}")
plt.tight_layout()
# chirps_image.shape, era5_image.shape

In [ ]:
n = 10_591
X = np.array(reprojected_era5[:n+1])
Y = chirps_images[:n+1]
# y = chirps_images

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, shuffle=False, random_state=69)